In [28]:
from torchvision import datasets
"""
ToTensor()是torchvision的图像变换，专门把PIL Image/numpy图片数组 -> 转换成PyTorch Tensor
"""
from torchvision.transforms import ToTensor
from torchvision import transforms

import torch
import torch.nn as nn
import numpy as np

In [29]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(device)

cpu


In [30]:
"""
ToTensor()是torchvision的图像变换，专门把PIL Image/numpy图片数组 -> 转换成PyTorch Tensor

transforms.Compose：把多个图像操作按顺序打包，依次执行

ToTensor():
        输入：PIL图片/numpy数组，像素范围[0, 255]，形状 [H, W, C]
        输出：torch张量，像素缩放至 [0, 1]，自动调换维度为 [C, H, W] （PyTorch卷积要求格式）
        这一步只是缩放到0～1，不是Z-score标准化
"""
transform = transforms.Compose([ToTensor(),  # 转换为tensor，进行归一化
                                # transforms.Normalize(mean, std)   标准化，mean和std是数据集的均值和方差
                                ])

In [31]:
train_ds = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

In [32]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True) 

In [33]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        # 继承父类初始化方法，子类有父类的属性
        super().__init__()
        # 自定义属性，实例化两个对象
        self.flatten = nn.Flatten()
        """
        nn.Sequential是PyTorch中用于快速搭建顺序（串行）神经网络的容器，它按照模块传入顺序，自动将前一个层的输出作为下一个层的输入。使用nn.Sequential可以极大地简化代码，避免手动在forward函数中逐层调用
        """
        self.linear_relu_stack = nn.Sequential(
            # in_features=784, out_features=300, 784是输入特征数，300是输出特征数
            nn.Linear(784, 300),
            nn.ReLU(),                  # 激活函数
            # 隐藏层神经元数100
            nn.Linear(300, 100),
            nn.ReLU(),                  # 激活函数
            # 输出层神经元数10
            nn.Linear(100, 10),
        )
        
    def forward(self, x):
        # x.shape [batch size, 1, 28, 28
        x = self.flatten(x)
        # 展平后 x.shape [batch size, 784]
        logits = self.linear_relu_stack(x)
        # logits.shape [batch size, 10]
        return logits       # 没有经过softmax，称为logits
    
# 实例化一个对象
model = NeuralNetwork()

In [34]:
# 1. 定义损失函数，采用交叉熵损失
loss_fct = nn.CrossEntropyLoss()

# 2. 定义优化器，采用SGD
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [35]:
from sklearn.metrics import accuracy_score

@torch.no_grad()        # 装饰器，禁止反向传播，节省内存
def evaluating(model, dataloader, loss_fct):
    loss_list = []
    pred_list = []
    label_list = []
    for datas, labels in dataloader:
        datas = datas.to(device)
        labels = labels.to(device)
        
        logits = model(datas)
        loss = loss_fct(logits, labels)
        loss_list.append(loss.item())
        
        preds = logits.argmax(axis=-1)
        pred_list.extend(preds.cpu().numpy().tolist())
        label_list.extend(labels.cpu().numpy().tolist())
    
    acc = accuracy_score(label_list, pred_list)
    return np.mean(loss_list), acc

In [36]:
evaluating(model, train_loader, loss_fct)

(np.float64(2.3062293435414634), 0.14363333333333334)